In [1]:

import os
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, TransformerConv
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import GNNExplainer
from torch_geometric.explain.config import ModelConfig, ModelReturnType, ModelMode

import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import networkx as nx
import shap
import random

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

OUTDIR = "./explanations"
ensure_dir(OUTDIR)

def train_model(model, data, optimizer, epochs=150, print_every=30):
    model.to(device)
    data = data.to(device)
    for epoch in range(1, epochs+1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        if epoch % print_every == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                out_eval = model(data.x, data.edge_index)
                preds = out_eval.argmax(dim=1)
                train_idx = data.train_mask.cpu().numpy()
                test_idx = data.test_mask.cpu().numpy()
                acc_train = (preds[train_idx] == data.y[train_idx]).float().mean().item()
                acc_test = (preds[test_idx] == data.y[test_idx]).float().mean().item()
                print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Train Acc: {acc_train:.4f} | Test Acc: {acc_test:.4f}")
    # final evaluation
    model.eval()
    with torch.no_grad():
        out_final = model(data.x, data.edge_index)
        preds = out_final.argmax(dim=1).cpu().numpy()
    return model, preds

def evaluate_model(model, data):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        preds = out.argmax(dim=1).cpu().numpy()
    test_idx = data.test_mask.cpu().numpy()
    print("Test classification report:")
    print(classification_report(data.y.cpu().numpy()[test_idx], preds[test_idx], target_names=le.classes_))
    return preds

# small plotting helper to save figure
def save_fig(fig, fname):
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)

C:\Users\WELCOME\anaconda3\envs\pregnancy_gnn\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1) Load + Preprocess (same as before)

df = pd.read_csv("Maternal Health Risk Data Set.csv")

# keep selected numeric features, drop rows with missing
features = ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']
df = df.dropna(subset=features + ['RiskLevel']).reset_index(drop=True)

# add/ensure timestamp
if 'Timestamp' not in df.columns:
    df['Timestamp'] = range(len(df))

# encode labels
le = LabelEncoder()
df['RiskLevel_encoded'] = le.fit_transform(df['RiskLevel'])

# sort by timestamp (temporal order)
df = df.sort_values('Timestamp').reset_index(drop=True)

# scale features
scaler = StandardScaler()
X = scaler.fit_transform(df[features])
y_np = df['RiskLevel_encoded'].values

# build temporal edge_index
k = 3
edge_list = []
for i in range(len(df)):
    for j in range(1, k+1):
        if i + j < len(df):
            edge_list.append([i, i + j])  # directed

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
x = torch.tensor(X, dtype=torch.float)
y = torch.tensor(y_np, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

# train/test split (80/20)
num_nodes = data.num_nodes
num_train = int(0.8 * num_nodes)
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[:num_train] = True
test_mask = ~train_mask
data.train_mask = train_mask
data.test_mask = test_mask

print("Nodes:", data.num_nodes, "Edges:", data.num_edges, "Classes:", len(np.unique(y_np)))

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Nodes: 1014 Edges: 3036 Classes: 3


In [3]:
# 2) Define models (GCN baseline & Transformer)
# Implement forward(x, edge_index) signature to work with Explainer

class GCNModel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        # capture penultimate embeddings for SHAP later
        self.embedding = x  # shape: [N, hidden_channels]
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

class TransformerModel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=2):
        super().__init__()
        # TransformerConv behaves like attention-based conv (can capture richer interactions)
        self.conv1 = TransformerConv(in_channels, hidden_channels // heads, heads=heads, concat=True)  # outputs hidden_channels
        self.conv2 = TransformerConv(hidden_channels, out_channels, heads=1, concat=False)  # outputs out_channels

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        self.embedding = x  # penultimate embeddings
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# instantiate
in_ch = data.num_node_features
n_classes = len(np.unique(y_np))

gcn = GCNModel(in_ch, hidden_channels=32, out_channels=n_classes).to(device)
transformer = TransformerModel(in_ch, hidden_channels=32, out_channels=n_classes).to(device)


In [4]:
# 3) Train both models

print("\nTraining GCN baseline...")
optimizer_gcn = torch.optim.Adam(gcn.parameters(), lr=0.01, weight_decay=5e-4)
gcn, preds_gcn = train_model(gcn, data, optimizer_gcn, epochs=160, print_every=40)

print("\nTraining TransformerConv model...")
optimizer_tf = torch.optim.Adam(transformer.parameters(), lr=0.01, weight_decay=5e-4)
transformer, preds_tf = train_model(transformer, data, optimizer_tf, epochs=160, print_every=40)

# evaluate & print reports
print("\nGCN Evaluation:")
preds_gcn = evaluate_model(gcn, data)

print("\nTransformer Evaluation:")
preds_tf = evaluate_model(transformer, data)


Training GCN baseline...
Epoch 001 | Loss: 1.0971 | Train Acc: 0.4217 | Test Acc: 0.4335
Epoch 040 | Loss: 0.9371 | Train Acc: 0.5561 | Test Acc: 0.7143
Epoch 080 | Loss: 0.9007 | Train Acc: 0.5906 | Test Acc: 0.7340
Epoch 120 | Loss: 0.8802 | Train Acc: 0.6116 | Test Acc: 0.7094
Epoch 160 | Loss: 0.8664 | Train Acc: 0.6165 | Test Acc: 0.6995

Training TransformerConv model...
Epoch 001 | Loss: 1.2688 | Train Acc: 0.3095 | Test Acc: 0.3695
Epoch 040 | Loss: 0.6579 | Train Acc: 0.7559 | Test Acc: 0.7882
Epoch 080 | Loss: 0.4792 | Train Acc: 0.8298 | Test Acc: 0.8079
Epoch 120 | Loss: 0.4189 | Train Acc: 0.8656 | Test Acc: 0.8128
Epoch 160 | Loss: 0.3850 | Train Acc: 0.8940 | Test Acc: 0.8177

GCN Evaluation:
Test classification report:
              precision    recall  f1-score   support

   high risk       0.97      0.61      0.75        59
    low risk       0.59      0.77      0.67        78
    mid risk       0.72      0.70      0.71        66

    accuracy                        

In [5]:
# 4) Explainer setup and per-node explanations saved to disk

# ModelConfig for multiclass node-level explanations
model_config = ModelConfig(
    mode=ModelMode.multiclass_classification,
    task_level='node',
    return_type=ModelReturnType.log_probs
)

# create explainer instances for both models (GNNExplainer)
explainer_gcn = Explainer(
    model=gcn,
    algorithm=GNNExplainer(epochs=200),
    explanation_type='model',
    node_mask_type='attributes',
    edge_mask_type='object',
    model_config=model_config,
)

explainer_tf = Explainer(
    model=transformer,
    algorithm=GNNExplainer(epochs=200),
    explanation_type='model',
    node_mask_type='attributes',
    edge_mask_type='object',
    model_config=model_config,
)

# pick a few test nodes to explain (limit for speed)
test_nodes = np.where(data.test_mask.cpu().numpy())[0]
random.seed(0)
explain_nodes = list(np.random.choice(test_nodes, size=min(6, len(test_nodes)), replace=False))
print("\nNodes to explain:", explain_nodes)

# For each node: compute explanation for both models, save PNGs (feature bar, subgraph edges)
for node_idx in explain_nodes:
    node_idx = int(node_idx)
    print(f"\nExplaining node {node_idx}...")

    # GCN explanation
    explanation_gcn = explainer_gcn(x=data.x, edge_index=data.edge_index, index=node_idx)
    node_mask_gcn = explanation_gcn.node_mask.detach().cpu().numpy()
    # node_mask_gcn might be full matrix [N, F] or [F]; pick row for node
    if node_mask_gcn.ndim == 2 and node_mask_gcn.shape[0] == data.num_nodes:
        node_mask_gcn = node_mask_gcn[node_idx]
    node_mask_gcn = node_mask_gcn.flatten()

    em_gcn = explanation_gcn.edge_mask.detach().cpu().numpy()
    ei_gcn = explanation_gcn.edge_index.detach().cpu().numpy()

    # Save feature importance bar
    fig = plt.figure(figsize=(6,3))
    plt.bar(features, node_mask_gcn)
    plt.title(f"GCN - Node {node_idx} feature importance | True: {le.inverse_transform([int(data.y[node_idx])])[0]}")
    plt.xticks(rotation=45)
    save_fig(fig, os.path.join(OUTDIR, f"gcn_node_{node_idx}_feat.png"))

    # Save subgraph with important edges (top-k)
    # Normalize edge mask
    emg = em_gcn
    emg = (emg - emg.min()) / (np.ptp(emg) + 1e-9)
    top_k = min(30, len(emg))
    top_idx = np.argsort(emg)[-top_k:]
    edges_local = ei_gcn[:, top_idx].T  # shape (k,2)
    G = nx.DiGraph()
    G.add_edges_from([(int(u), int(v)) for u,v in edges_local])
    fig = plt.figure(figsize=(6,5))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_size=120, arrows=True)
    # thicker edges for higher importance
    weights = emg[top_idx]
    for (u,v), w in zip(list(G.edges()), weights):
        nx.draw_networkx_edges(G, pos, edgelist=[(u,v)], width=1+4*w)
    plt.title(f"GCN - Node {node_idx} top explanatory edges")
    save_fig(fig, os.path.join(OUTDIR, f"gcn_node_{node_idx}_edges.png"))

    # Transformer explanation
    explanation_tf = explainer_tf(x=data.x, edge_index=data.edge_index, index=node_idx)
    node_mask_tf = explanation_tf.node_mask.detach().cpu().numpy()
    if node_mask_tf.ndim == 2 and node_mask_tf.shape[0] == data.num_nodes:
        node_mask_tf = node_mask_tf[node_idx]
    node_mask_tf = node_mask_tf.flatten()

    em_tf = explanation_tf.edge_mask.detach().cpu().numpy()
    ei_tf = explanation_tf.edge_index.detach().cpu().numpy()

    fig = plt.figure(figsize=(6,3))
    plt.bar(features, node_mask_tf)
    plt.title(f"Transformer - Node {node_idx} feature importance | True: {le.inverse_transform([int(data.y[node_idx])])[0]}")
    plt.xticks(rotation=45)
    save_fig(fig, os.path.join(OUTDIR, f"tf_node_{node_idx}_feat.png"))

    # TF top edges
    emt = em_tf
    emt = (emt - emt.min()) / (np.ptp(emt) + 1e-9)
    top_k_tf = min(30, len(emt))
    top_idx_tf = np.argsort(emt)[-top_k_tf:]
    edges_local_tf = ei_tf[:, top_idx_tf].T
    G2 = nx.DiGraph()
    G2.add_edges_from([(int(u), int(v)) for u,v in edges_local_tf])
    fig = plt.figure(figsize=(6,5))
    pos = nx.spring_layout(G2, seed=42)
    nx.draw(G2, pos, with_labels=True, node_size=120, arrows=True)
    weights_tf = emt[top_idx_tf]
    for (u,v), w in zip(list(G2.edges()), weights_tf):
        nx.draw_networkx_edges(G2, pos, edgelist=[(u,v)], width=1+4*w)
    plt.title(f"Transformer - Node {node_idx} top explanatory edges")
    save_fig(fig, os.path.join(OUTDIR, f"tf_node_{node_idx}_edges.png"))

    # Side-by-side comparison saved (feature bars)
    fig, axs = plt.subplots(1,2, figsize=(12,3))
    axs[0].bar(features, node_mask_gcn); axs[0].set_title("GCN feature importance")
    axs[1].bar(features, node_mask_tf); axs[1].set_title("Transformer feature importance")
    for ax in axs:
        ax.tick_params(axis='x', rotation=45)
    save_fig(fig, os.path.join(OUTDIR, f"compare_node_{node_idx}_feat.png"))

    # Side-by-side edges: we'll plot both graphs smaller
    fig, axs = plt.subplots(1,2, figsize=(12,5))
    pos1 = nx.spring_layout(G, seed=2)
    nx.draw(G, pos1, ax=axs[0], with_labels=True, node_size=80, arrows=True)
    axs[0].set_title("GCN top edges")
    pos2 = nx.spring_layout(G2, seed=2)
    nx.draw(G2, pos2, ax=axs[1], with_labels=True, node_size=80, arrows=True)
    axs[1].set_title("Transformer top edges")
    save_fig(fig, os.path.join(OUTDIR, f"compare_node_{node_idx}_edges.png"))

print(f"\nSaved explainer PNGs to {OUTDIR}")


Nodes to explain: [np.int64(921), np.int64(961), np.int64(1004), np.int64(882), np.int64(896), np.int64(992)]

Explaining node 921...


C:\Users\WELCOME\AppData\Local\Temp\ipykernel_14524\3706901758.py:75: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
C:\Users\WELCOME\AppData\Local\Temp\ipykernel_14524\3706901758.py:75: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



Explaining node 961...


C:\Users\WELCOME\AppData\Local\Temp\ipykernel_14524\3706901758.py:75: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
C:\Users\WELCOME\AppData\Local\Temp\ipykernel_14524\3706901758.py:75: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



Explaining node 1004...


...


Explaining node 896...


C:\Users\WELCOME\AppData\Local\Temp\ipykernel_14524\3706901758.py:75: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
C:\Users\WELCOME\AppData\Local\Temp\ipykernel_14524\3706901758.py:75: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



Explaining node 992...


C:\Users\WELCOME\AppData\Local\Temp\ipykernel_14524\3706901758.py:75: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
C:\Users\WELCOME\AppData\Local\Temp\ipykernel_14524\3706901758.py:75: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



Saved explainer PNGs to ./explanations


In [6]:
# 5) SHAP on node embeddings
# Compute embeddings for nodes from both models (penultimate activations),
# and use KernelExplainer to explain the embedding-to-prediction mapping for selected nodes.

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# helper to get embedding + model output probability for a node after replacing its features
def get_embedding_and_logits(model, data):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)  # also sets model.embedding
        # model.embedding is tensor [N, hidden]
        emb = model.embedding.detach().cpu().numpy()
        logits = out.detach().cpu().numpy()   # log probs
        probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    return emb, probs

# compute embeddings
emb_gcn, probs_gcn = get_embedding_and_logits(gcn, data)
emb_tf, probs_tf = get_embedding_and_logits(transformer, data)

# function to wrap for SHAP: when explaining node i,
# we vary only that node's feature vector while keeping other node features fixed.
def make_shap_predict_fn(model, data, node_idx, class_index):
    # model: trained model (on device)
    # returns a function f(X_variations) -> probability for class_index for node_idx
    data_cpu = Data(x=data.x.clone().cpu(), edge_index=data.edge_index.clone().cpu(), y=data.y.clone().cpu())
    def predict_fn(X):  # X shape: (m, num_features)
        # For each row of X, replace data_cpu.x[node_idx] with that row, run model, and return probability of class_index for node_idx
        outputs = []
        for row in X:
            x_mod = data_cpu.x.clone()
            x_mod[node_idx] = torch.tensor(row, dtype=torch.float)
            # run model on CPU to avoid device issues
            model_cpu = model.to('cpu')
            model_cpu.eval()
            with torch.no_grad():
                logits = model_cpu(x_mod, data_cpu.edge_index)
                probs = torch.exp(logits)  # because logits are log_softmax -> exp gives probs
                outputs.append(probs[node_idx, class_index].item())
            model_cpu.to(device)  # return model to original device (best-effort)
        return np.array(outputs)
    return predict_fn

# SHAP setup: choose small background and limited shap nodes for compute
bg_size = min(50, data.num_nodes)
bg_idx = np.random.choice(np.where(data.train_mask.cpu().numpy())[0], size=bg_size, replace=False)
background = data.x.cpu().numpy()[bg_idx]  # shape [bg_size, num_features]

# pick nodes to compute SHAP (same as explain_nodes, but limit if too many)
shap_nodes = explain_nodes[:4]  # up to 4 nodes for speed
print("\nSHAP will be computed for nodes:", shap_nodes)

for node_idx in shap_nodes:
    node_idx = int(node_idx)
    true_label = int(data.y[node_idx].item())
    # choose class_index to explain: predicted class by model or true_label
    # We'll explain predicted class for each model separately

    # GCN SHAP
    pred_class_gcn = int(np.argmax(probs_gcn[node_idx]))
    predict_fn_gcn = make_shap_predict_fn(gcn, data, node_idx, pred_class_gcn)
    # KernelExplainer: background is a subset; this will be slow but workable for small feature dimension (6)
    explainer_shap_gcn = shap.KernelExplainer(predict_fn_gcn, background)
    # explain this node's feature vector
    shap_values_gcn = explainer_shap_gcn.shap_values(data.x.cpu().numpy()[node_idx], nsamples=200)  # nsamples small for speed
    # shap_values_gcn shape: list of arrays if multiclass -> we used class pred, Kernel returns vector
    # We'll get per-feature contributions; ensure array
    if isinstance(shap_values_gcn, list):
        shap_vals = np.array(shap_values_gcn).T.squeeze()  # try to reduce
    else:
        shap_vals = np.array(shap_values_gcn).flatten()

    # Save SHAP bar
    fig = plt.figure(figsize=(6,3))
    plt.bar(features, shap_vals)
    plt.title(f"SHAP on embeddings (GCN) - Node {node_idx} | pred class {pred_class_gcn}")
    plt.xticks(rotation=45)
    save_fig(fig, os.path.join(OUTDIR, f"shap_gcn_node_{node_idx}.png"))

    # Transformer SHAP
    pred_class_tf = int(np.argmax(probs_tf[node_idx]))
    predict_fn_tf = make_shap_predict_fn(transformer, data, node_idx, pred_class_tf)
    explainer_shap_tf = shap.KernelExplainer(predict_fn_tf, background)
    shap_values_tf = explainer_shap_tf.shap_values(data.x.cpu().numpy()[node_idx], nsamples=200)
    if isinstance(shap_values_tf, list):
        shap_vals_tf = np.array(shap_values_tf).T.squeeze()
    else:
        shap_vals_tf = np.array(shap_values_tf).flatten()

    fig = plt.figure(figsize=(6,3))
    plt.bar(features, shap_vals_tf)
    plt.title(f"SHAP on embeddings (Transformer) - Node {node_idx} | pred class {pred_class_tf}")
    plt.xticks(rotation=45)
    save_fig(fig, os.path.join(OUTDIR, f"shap_tf_node_{node_idx}.png"))

    # Save comparison (GNNExplainer vs SHAP side-by-side for feature attributions)
    # Load earlier node_mask arrays (we have them in node_mask_gcn/node_mask_tf if run above in same loop)
    # For safety recompute explainer-based masks quickly:
    exp_g = explainer_gcn(x=data.x, edge_index=data.edge_index, index=node_idx)
    nm_g = exp_g.node_mask.detach().cpu().numpy()
    if nm_g.ndim == 2 and nm_g.shape[0] == data.num_nodes:
        nm_g = nm_g[node_idx]
    nm_g = nm_g.flatten()

    exp_t = explainer_tf(x=data.x, edge_index=data.edge_index, index=node_idx)
    nm_t = exp_t.node_mask.detach().cpu().numpy()
    if nm_t.ndim == 2 and nm_t.shape[0] == data.num_nodes:
        nm_t = nm_t[node_idx]
    nm_t = nm_t.flatten()

    fig, axs = plt.subplots(2,2, figsize=(12,8))
    axs[0,0].bar(features, nm_g); axs[0,0].set_title("GNNExplainer (GCN)")
    axs[0,1].bar(features, nm_t); axs[0,1].set_title("GNNExplainer (Transformer)")
    axs[1,0].bar(features, shap_vals); axs[1,0].set_title("SHAP (GCN)")
    axs[1,1].bar(features, shap_vals_tf); axs[1,1].set_title("SHAP (Transformer)")
    for axrow in axs:
        for ax in axrow:
            ax.tick_params(axis='x', rotation=45)
    save_fig(fig, os.path.join(OUTDIR, f"compare_all_node_{node_idx}.png"))

print("\nSaved SHAP and comparison PNGs to", OUTDIR)
print("Done. Note: SHAP KernelExplainer uses sampling and can be slow; reduce nsamples/background for speed.")


SHAP will be computed for nodes: [np.int64(921), np.int64(961), np.int64(1004), np.int64(882)]

Saved SHAP and comparison PNGs to ./explanations
Done. Note: SHAP KernelExplainer uses sampling and can be slow; reduce nsamples/background for speed.
